In [1]:
import numpy as np
import pandas as pd

In [11]:
interactions = pd.read_csv(
    "../data/db/banner_interactions.csv",
    parse_dates=["event_date"]
)
banners = pd.read_csv(
    "../data/db/banners.csv",
    parse_dates=["created_at"]
)

users = pd.read_csv("../data/db/users.csv")

In [6]:

TRAIN_END = pd.Timestamp("2026-02-28")
VALID_END = pd.Timestamp("2026-03-15")

# Делим данные по времени: модель видит только прошлое.
train_df = interactions[interactions["event_date"] <= TRAIN_END].copy()
valid_df = interactions[
    (interactions["event_date"] > TRAIN_END)
    & (interactions["event_date"] <= VALID_END)
].copy()
test_df = interactions[interactions["event_date"] > VALID_END].copy()

In [8]:
train_df.head()

,event_date,user_id,banner_id,impressions,clicks,ctr
0,2026-01-01,u_00007,b_0082,3,0,0.0
1,2026-01-01,u_00009,b_0234,3,0,0.0
2,2026-01-01,u_00012,b_0041,4,0,0.0
3,2026-01-01,u_00012,b_0215,5,1,0.2
4,2026-01-01,u_00015,b_0210,5,2,0.4


In [9]:
valid_df.head()

,event_date,user_id,banner_id,impressions,clicks,ctr
98282,2026-03-01,u_00004,b_0281,5,0,0.0
98283,2026-03-01,u_00007,b_0147,1,0,0.0
98284,2026-03-01,u_00011,b_0174,2,0,0.0
98285,2026-03-01,u_00015,b_0159,2,0,0.0
98286,2026-03-01,u_00024,b_0181,3,0,0.0


In [10]:
test_df.head()

,event_date,user_id,banner_id,impressions,clicks,ctr
123117,2026-03-16,u_00001,b_0209,6,0,0.0
123118,2026-03-16,u_00003,b_0063,8,0,0.0
123119,2026-03-16,u_00003,b_0241,5,0,0.0
123120,2026-03-16,u_00007,b_0151,1,0,0.0
123121,2026-03-16,u_00008,b_0274,1,0,0.0


In [12]:
user_ids = users["user_id"].drop_duplicates().tolist()
banner_ids = banners["banner_id"].drop_duplicates().tolist()

In [14]:
user2idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
item2idx = {banner_id: idx for idx, banner_id in enumerate(banner_ids)}
idx2item = {idx: banner_id for banner_id, idx in item2idx.items()}


In [17]:
item2idx

{'b_0001': 0,
 'b_0002': 1,
 'b_0003': 2,
 'b_0004': 3,
 'b_0005': 4,
 'b_0006': 5,
 'b_0007': 6,
 'b_0008': 7,
 'b_0009': 8,
 'b_0010': 9,
 'b_0011': 10,
 'b_0012': 11,
 'b_0013': 12,
 'b_0014': 13,
 'b_0015': 14,
 'b_0016': 15,
 'b_0017': 16,
 'b_0018': 17,
 'b_0019': 18,
 'b_0020': 19,
 'b_0021': 20,
 'b_0022': 21,
 'b_0023': 22,
 'b_0024': 23,
 'b_0025': 24,
 'b_0026': 25,
 'b_0027': 26,
 'b_0028': 27,
 'b_0029': 28,
 'b_0030': 29,
 'b_0031': 30,
 'b_0032': 31,
 'b_0033': 32,
 'b_0034': 33,
 'b_0035': 34,
 'b_0036': 35,
 'b_0037': 36,
 'b_0038': 37,
 'b_0039': 38,
 'b_0040': 39,
 'b_0041': 40,
 'b_0042': 41,
 'b_0043': 42,
 'b_0044': 43,
 'b_0045': 44,
 'b_0046': 45,
 'b_0047': 46,
 'b_0048': 47,
 'b_0049': 48,
 'b_0050': 49,
 'b_0051': 50,
 'b_0052': 51,
 'b_0053': 52,
 'b_0054': 53,
 'b_0055': 54,
 'b_0056': 55,
 'b_0057': 56,
 'b_0058': 57,
 'b_0059': 58,
 'b_0060': 59,
 'b_0061': 60,
 'b_0062': 61,
 'b_0063': 62,
 'b_0064': 63,
 'b_0065': 64,
 'b_0066': 65,
 'b_0067': 66,
 'b_0